# Camada Bronze — `matriculas`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Descrição da tabela:** Histórico de matrículas por faculdade, curso e semestre  
**Arquivo bruto:** `Atividade11 - matriculas.txt`

In [1]:
import json
import sys
from pathlib import Path

# Garante que notebooks/ esteja no sys.path independente do diretório de execução
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.bronze_utils import (
    get_project_paths,
    ingest_raw_file,
    profile_bronze_table,
    save_bronze_outputs,
)

ROOT_DIR, RAW_DIR, BRONZE_DIR = get_project_paths()
print(f"Origem: {RAW_DIR / 'Atividade11 - matriculas.txt'}")
print(f"Destino Bronze: {BRONZE_DIR}")

Origem: C:\Projects\trabalho-final-mineacao-de-dados\Bases TRabalho Final\Atividade11 - matriculas.txt
Destino Bronze: C:\Projects\trabalho-final-mineacao-de-dados\data\bronze


In [2]:
df, meta = ingest_raw_file(
    dataset_name='matriculas',
    filename='Atividade11 - matriculas.txt',
    sep='|',
    mixed_sep=False,
    raw_dir=RAW_DIR,
)

profile = profile_bronze_table(df, meta)
parquet_path, profile_path = save_bronze_outputs(df, profile, 'matriculas', BRONZE_DIR)

print(f"Linhas ingeridas: {meta['rows']:,} | Tamanho bruto: {meta['size_mb']} MB")
print(f"IDs únicos de aluno: {profile['unique_id_aluno']:,} | Duplicatas exatas: {profile['exact_duplicates']:,}")
print(f"Salvo em: {parquet_path.name} e {profile_path.name}")
df.head(5)

Linhas ingeridas: 700,000 | Tamanho bruto: 23.9468 MB
IDs únicos de aluno: 376,925 | Duplicatas exatas: 0
Salvo em: matriculas.parquet e matriculas_profile.json


,id_matricula,id_aluno,faculdade,curso,semestre,status,_bronze_source_file,_bronze_ingested_at
0,1,230807,UNESP,Medicina,9,Evadido,Atividade11 - matriculas.txt,2026-11-13T18:54:55Z
1,2,382041,UNICAMP,Direito,7,Ativo,Atividade11 - matriculas.txt,2026-11-13T18:54:55Z
2,3,450121,USP,Engenharia,8,Ativo,Atividade11 - matriculas.txt,2026-11-13T18:54:55Z
3,4,249182,UNICAMP,ADM,5,Ativo,Atividade11 - matriculas.txt,2026-11-13T18:54:55Z
4,5,32594,USP,ADM,2,Ativo,Atividade11 - matriculas.txt,2026-11-13T18:54:55Z


In [3]:
print("=== Campos vazios por coluna ===")
print(json.dumps(profile["empty_counts"], ensure_ascii=False, indent=2))
if profile["anomalies"]:
    print("\n=== Anomalias detectadas para tratamento na Silver ===")
    print(json.dumps(profile["anomalies"], ensure_ascii=False, indent=2))

=== Campos vazios por coluna ===
{
  "id_matricula": 0,
  "id_aluno": 0,
  "faculdade": 0,
  "curso": 0,
  "semestre": 0,
  "status": 0
}

=== Anomalias detectadas para tratamento na Silver ===
{
  "status_distribuicao": {
    "Ativo": 489938,
    "Evadido": 105176,
    "Trancado": 104886
  }
}
